In [1]:
#Importing libraries
import numpy as np
import scipy as sp
from scipy.optimize import minimize

In [22]:
#Constructing the function
def f(p):
    #a0=1.04
    a0=np.random.normal(1.04,0.29)
    #r=0.96
    r=np.random.normal(0.96,0.18)
    pcosdelta=1/a0+0.5*r*(p**2)
    return pcosdelta
    #den=pcosdelta-1j*p
    #return den

In [40]:
#Getting data points
p=np.linspace(1,10,10)
fp=[]
for i in p:
    fp.append(f(i))

In [4]:
#Defining least fit norms

def l2(p,fv,g):
    l2=0
    for i in range(len(fv)):
        l2=l2+(fv[i]-g(p[i]))**2
    return l2

def l1(p,fv,g):
    l1=0
    for i in range(len(fv)):
        l1=l1+(fv[i]-g(p[i]))
    return l1    

def linf(p,fv,g):
    linf=0
    err=[]
    for i in range(len(fv)):
        err.append(fv[i]-g(p[i]))
    i=np.argmax(err)
    linf=err[i]
    return linf

In [5]:
# Define the L2 norm function for optimization for constant function
def l2(c, p, fv):
    cr, ci = c[0], c[1]
    sum = 0
    for i in range(len(fv)):
        sum=sum+np.abs(fv[i] - (cr + ci * 1j))**2
    return sum
    
a0 = [0, 0]

# Minimize the L2 norm function
res = minimize(l2, a0, args=(p, fp))

coeff=[]
for i in range(2):
    coeff.append(res.x[i])

kaisq=l2(a0,p,fp)
kaisq

6223.323658215648

In [52]:
#Constructing Pade Approximant using polynomial fit 
def f(p):
    #a0=1.04
    a0=np.random.normal(1.04,0.29)
    #r=0.96
    r=np.random.normal(0.96,0.18)
    pcosdelta=1/a0+0.5*r*(p**2)
    den=pcosdelta-1j*p
    return den

#1. Getting data points
p=np.linspace(1,10,10)
fp=[]
for i in p:
    fp.append(f(i))
# 2. Finding the best numerator least fit
coeff=0
kaisqo=10e6
kaisq=10e5
num=5
for i in range(5):
    coeffo=coeff
    coeff=np.polyfit(p,fp,i)
    poly=np.poly1d(coeff)
    kaisqo=kaisq
    kaisq=0
    for j in range(num):
        kaisq=kaisq+(fp[j]-poly(p[j]))**2   
    if (kaisqo<=kaisq):
        coeff=coeffo
        break
print(coeff)
poly=np.poly1d(coeff)

[ 0.28680248  1.48397759 -0.63847027]


array([-5.57362561,  0.39941096])

In [74]:
#Trying averaging of zeroes
r1,r2=[],[]
for i in range (10):
    #1.Obtaining the roots 
    p=np.linspace(1,10,10)
    fp=[]
    for i in p:
        fp.append(f(i))
    coeff=0
    kaisqo=10e6
    kaisq=10e5
    num=5
    for i in range(5):
        coeffo=coeff
        coeff=np.polyfit(p,fp,i)
        poly=np.poly1d(coeff)
        kaisqo=kaisq
        kaisq=0
        for j in range(num):
            kaisq=kaisq+(fp[j]-poly(p[j]))**2   
        if (kaisqo<=kaisq):
            coeff=coeffo
            break
    poly=np.poly1d(coeff)
    roots=np.roots(poly)
    r1.append(roots[0])
    for i in range(1,len(roots)):
        if (roots[i].imag==-roots[i-1].imag):
            r2.append(roots[i])
        else:
            r1.append(roots[i])
avgr1=sum(r1)/len(r1)
std1=np.std(r1)
print(avgr1,std1)


avgr2=sum(r2)/len(r2)
std2=np.std(r2)
print(avgr2,std2)

(6.210626630083831+1.6239764708839055j) 26.78576069742969
(1.8412353352066004-2.657416043264573j) 3.242359462334183


In [91]:
import numpy as np
from scipy.optimize import minimize

# Constructing Pade Approximant using polynomial fit
def f(p):
    a0 = np.random.normal(1.04, 0.29)
    r = np.random.normal(0.96, 0.18)
    pcosdelta = 1 / a0 + 0.5 * r * (p**2)
    den = pcosdelta - 1j * p
    return den

# 1. Getting data points
p = np.linspace(1, 10, 10)
fp = np.array([f(i) for i in p])

# 2. Define the error function for L1 norm
def l1_error(coeff, p, fp, m, n):
    num_coeff_real = coeff[:m + 1]
    num_coeff_imag = coeff[m + 1:2 * (m + 1)]
    den_coeff_real = [1.0] + coeff[2 * (m + 1):2 * (m + 1) + n]  # Fix constant term to 1
    den_coeff_imag = [0.0] + coeff[2 * (m + 1) + n:]  # Fix imaginary part of constant term to 0

    def pade_approx(x):
        num = np.poly1d(num_coeff_real + 1j * num_coeff_imag)(x)
        den = np.poly1d(den_coeff_real + 1j * den_coeff_imag)(x)
        return num / den

    error = 0
    for i in range(len(p)):
        error += np.abs(fp[i] - pade_approx(p[i]))
    return error

# Initial guess for coefficients (real and imaginary parts)
m = 3  # Degree of numerator
n = 3  # Degree of denominator
initial_guess = np.random.randn(2 * (m + 1 + n))

# Optimize the coefficients using L1 norm
res = minimize(l1_error, initial_guess, args=(p, fp, m, n), method='Nelder-Mead')

# Extract optimized coefficients
opt_coeff = res.x
num_coeff_real = opt_coeff[:m + 1]
num_coeff_imag = opt_coeff[m + 1:2 * (m + 1)]
den_coeff_real = [1.0] + opt_coeff[2 * (m + 1):2 * (m + 1) + n]
den_coeff_imag = [0.0] + opt_coeff[2 * (m + 1) + n:]

# Normalize the coefficients to ensure the constant term of the denominator is 1
denominator_poly = np.poly1d(den_coeff_real + 1j * den_coeff_imag)
denominator_const_term = denominator_poly[0]  # The constant term of the denominator

numerator_poly = np.poly1d(num_coeff_real + 1j * num_coeff_imag) / denominator_const_term
denominator_poly = denominator_poly / denominator_const_term

# Displaying the Pade approximant
print("Pade Approximant:")
print("Numerator:", numerator_poly)
print("Denominator:", denominator_poly)

# Evaluating the Pade approximant at points p
pade_approx_values = numerator_poly(p) / denominator_poly(p)
print("Pade Approximant values:", pade_approx_values)
print("Original function values:", fp)


Pade Approximant:
Numerator:                       3                        2
(-0.04461 + -1.016j) x + (-0.4607 + -0.2587j) x + (-0.283 + -0.6641j) x + (0.502 + -0.7931j)
Denominator:                        2
(-0.04247 + 0.09142j) x + (0.4744 + -2.707j) x + 1
Pade Approximant values: [ 0.75736448 -0.52404233j  1.84705277 -1.09914301j
  3.9316617  -1.8799849j   7.07817059 -2.83523785j
 11.41387265 -3.94413001j 17.09782838 -5.17575689j
 24.31968637 -6.48326602j 33.30352145 -7.79740385j
 44.31354641 -9.01768406j 57.6609559 -10.00000939j]
Original function values: [ 1.55681054 -1.j  3.08958786 -2.j  5.6282701  -3.j  9.43579219 -4.j
  9.26564407 -5.j 18.2744146  -6.j 28.54455213 -7.j 29.36756535 -8.j
 45.5580467  -9.j 57.66086011-10.j]


In [92]:
import numpy as np
from scipy.optimize import minimize

# Constructing Pade Approximant using polynomial fit
def f(p):
    a0 = np.random.normal(1.04, 0.29)
    r = np.random.normal(0.96, 0.18)
    pcosdelta = 1 / a0 + 0.5 * r * (p**2)
    den = pcosdelta - 1j * p
    return den

# 1. Getting data points
p = np.linspace(1, 10, 10)
fp = np.array([f(i) for i in p])

# 2. Define the error function for L2 norm
def l2_error(coeff, p, fp, m, n):
    num_coeff_real = coeff[:m + 1]
    num_coeff_imag = coeff[m + 1:2 * (m + 1)]
    den_coeff_real = [1.0] + coeff[2 * (m + 1):2 * (m + 1) + n]  # Fix constant term to 1
    den_coeff_imag = [0.0] + coeff[2 * (m + 1) + n:]  # Fix imaginary part of constant term to 0

    def pade_approx(x):
        num = np.poly1d(num_coeff_real + 1j * num_coeff_imag)(x)
        den = np.poly1d(den_coeff_real + 1j * den_coeff_imag)(x)
        return num / den

    error = 0
    for i in range(len(p)):
        error += np.abs(fp[i] - pade_approx(p[i]))**2
    return error

# Initial guess for coefficients (real and imaginary parts)
m = 3  # Degree of numerator
n = 3  # Degree of denominator
initial_guess = np.random.randn(2 * (m + 1 + n))

# Optimize the coefficients using L2 norm
res = minimize(l2_error, initial_guess, args=(p, fp, m, n), method='Nelder-Mead')

# Extract optimized coefficients
opt_coeff = res.x
num_coeff_real = opt_coeff[:m + 1]
num_coeff_imag = opt_coeff[m + 1:2 * (m + 1)]
den_coeff_real = [1.0] + opt_coeff[2 * (m + 1):2 * (m + 1) + n]
den_coeff_imag = [0.0] + opt_coeff[2 * (m + 1) + n:]

# Normalize the coefficients to ensure the constant term of the denominator is 1
denominator_poly = np.poly1d(den_coeff_real + 1j * den_coeff_imag)
denominator_const_term = denominator_poly[0]  # The constant term of the denominator

numerator_poly = np.poly1d(num_coeff_real + 1j * num_coeff_imag) / denominator_const_term
denominator_poly = denominator_poly / denominator_const_term

# Displaying the Pade approximant
print("Pade Approximant:")
print("Numerator:", numerator_poly)
print("Denominator:", denominator_poly)

# Evaluating the Pade approximant at points p
pade_approx_values = numerator_poly(p) / denominator_poly(p)
print("Pade Approximant values:", pade_approx_values)
print("Original function values:", fp)

Pade Approximant:
Numerator:                     3                       2
(0.2555 + 0.1145j) x + (-0.5959 + -1.171j) x + (0.7036 + 0.9037j) x + (0.1528 + 0.4755j)
Denominator:                       2
(0.03463 + 0.02511j) x + (-0.04917 + -0.1517j) x + 1
Pade Approximant values: [ 0.4738314  +0.38812559j  1.39890015 -1.15535516j
  3.83570788 -2.90626763j  7.84497693 -4.20759758j
 13.19775621 -5.09067583j 19.54816969 -5.84595431j
 26.52020518 -6.70686982j 33.80049092 -7.76547185j
 41.17991095 -9.01552637j 48.54208171-10.4119986j ]
Original function values: [ 1.40105538 -1.j  3.41080774 -2.j  4.61111    -3.j  6.4277913  -4.j
 14.21319069 -5.j 18.34563442 -6.j 19.67833343 -7.j 42.8913701  -8.j
 42.65722678 -9.j 45.02006031-10.j]


In [93]:
import numpy as np
from scipy.optimize import minimize

# Constructing Pade Approximant using polynomial fit
def f(p):
    a0 = np.random.normal(1.04, 0.29)
    r = np.random.normal(0.96, 0.18)
    pcosdelta = 1 / a0 + 0.5 * r * (p**2)
    den = pcosdelta - 1j * p
    return den

# 1. Getting data points
p = np.linspace(1, 10, 10)
fp = np.array([f(i) for i in p])

# 2. Define the error function for Minimax (Chebyshev) norm
def minimax_error(coeff, p, fp, m, n):
    num_coeff_real = coeff[:m + 1]
    num_coeff_imag = coeff[m + 1:2 * (m + 1)]
    den_coeff_real = [1.0] + coeff[2 * (m + 1):2 * (m + 1) + n]
    den_coeff_imag = [0.0] + coeff[2 * (m + 1) + n:]

    def pade_approx(x):
        num = np.poly1d(num_coeff_real + 1j * num_coeff_imag)(x)
        den = np.poly1d(den_coeff_real + 1j * den_coeff_imag)(x)
        return num / den

    error = np.max([np.abs(fp[i] - pade_approx(p[i])) for i in range(len(p))])
    return error

# Initial guess for coefficients (real and imaginary parts)
m = 3  # Degree of numerator
n = 3  # Degree of denominator
initial_guess = np.random.randn(2 * (m + 1 + n))

# Optimize the coefficients using Minimax norm
res = minimize(minimax_error, initial_guess, args=(p, fp, m, n), method='Nelder-Mead')

# Extract optimized coefficients
opt_coeff = res.x
num_coeff_real = opt_coeff[:m + 1]
num_coeff_imag = opt_coeff[m + 1:2 * (m + 1)]
den_coeff_real = [1.0] + opt_coeff[2 * (m + 1):2 * (m + 1) + n]
den_coeff_imag = [0.0] + opt_coeff[2 * (m + 1) + n:]

# Normalize the coefficients to ensure the constant term of the denominator is 1
denominator_poly = np.poly1d(den_coeff_real + 1j * den_coeff_imag)
denominator_const_term = denominator_poly[0]  # The constant term of the denominator

numerator_poly = np.poly1d(num_coeff_real + 1j * num_coeff_imag) / denominator_const_term
denominator_poly = denominator_poly / denominator_const_term

# Displaying the Pade approximant
print("Pade Approximant (Minimax Norm):")
print("Numerator:", numerator_poly)
print("Denominator:", denominator_poly)

# Evaluating the Pade approximant at points p
pade_approx_values = numerator_poly(p) / denominator_poly(p)
print("Pade Approximant values:", pade_approx_values)
print("Original function values:", fp)


Pade Approximant (Minimax Norm):
Numerator:                      3                      2
(0.3599 + -0.9539j) x + (0.481 + -0.1902j) x + (-1.579 + 0.776j) x + (-4.745 + -0.224j)
Denominator:                        2
(0.006148 + -0.1049j) x + (0.875 + -0.9464j) x + 1
Pade Approximant values: [-2.08711393-1.48105445j  0.59263411-2.05193499j  4.04710356-2.7311587j
  8.33112798-3.41258369j 13.35185871-4.00514895j 18.99880897-4.45577993j
 25.17090419-4.73727983j 31.78178755-4.83865353j 38.75948983-4.75878929j
 46.04461273-4.50243532j]
Original function values: [ 1.16507815 -1.j  3.37927955 -2.j  4.97438299 -3.j  7.99608949 -4.j
 13.81049483 -5.j 25.41274915 -6.j 18.21109479 -7.j 34.42914957 -8.j
 36.77187252 -9.j 50.87530965-10.j]
